# 🚀 Sistema RAG Completo con Groq (GRATIS y Rápido)
**Arquitectura explicada paso a paso**

**Pasos RAG:**
1. **Carga** → Lee documentos
2. **Divide** → Chunks manejables
3. **Incrusta** → Texto → vectores
4. **Almacena** → Vector DB
5. **Recupera** → Chunks relevantes
6. **Genera** → LLM + contexto

## 1. INSTALACIÓN Y CONFIGURACIÓN

In [61]:
%pip install langchain-classic langchain-groq langchain-chroma faiss-cpu python-dotenv langchain-huggingface beautifulsoup4 sentence-transformers chromadb pinecone-client cohere pymupdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\macdu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [62]:
from dotenv import load_dotenv

# Crea un archivo .env en la raíz del proyecto con tu API key:
#   GROQ_API_KEY=gsk_xxxxxxxxxxxx
# Obtén una clave gratuita en: https://console.groq.com/keys
# Para LangSmith (evaluación), añade también:
#   LANGSMITH_API_KEY=lsv2_xxxxxxxxxxxx  → https://smith.langchain.com
#   LANGSMITH_PROJECT=mi-rag-groq
load_dotenv()
print('✅ .env cargado')

✅ .env cargado


## 2. CARGA DE DOCUMENTOS

In [63]:
from langchain_community.document_loaders import WebBaseLoader, PyMuPDFLoader
import json

# Carga páginas web como documentos. Alternativas habituales:
#   TextLoader('archivo.txt')    → texto plano
#   PyMuPDFLoader('doc.pdf')     → PDFs  (pip install pymupdf)
#   CSVLoader('datos.csv')       → tablas

# Leer URLs desde fichero externo
with open("urls.json", "r", encoding="utf-8") as f:
    data = json.load(f)

urls = data["news"]
loader = WebBaseLoader(urls)
# pdf_loader = PyMuPDFLoader("Blood-Meridian.pdf")

url_docs = loader.load()
# pdf_docs = pdf_loader.load()
print(f"📄 Cargados {len(url_docs)} documentos web")
# print(f"📄 Cargados {len(pdf_docs)} documentos PDF")

# docs = url_docs + pdf_docs
docs = url_docs
print(f"📄 Cargados {len(docs)} documentos totales")

📄 Cargados 4 documentos web
📄 Cargados 4 documentos totales


## 3. DIVISIÓN EN FRAGMENTOS (CHUNKS)

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Divide documentos largos en chunks que quepan en el contexto del LLM.
# chunk_size=1000   → ~750 tokens por chunk (margen de seguridad)
# chunk_overlap=200 → los chunks se solapan 200 chars para no perder
#                     contexto en los cortes (ej: una frase a caballo)
# separators        → corta preferentemente en párrafos, luego líneas,
#                     luego frases; nunca a mitad de palabra
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?"]
)
splits = text_splitter.split_documents(docs)
print(f"✂️ Divididos en {len(splits)} chunks")
print(f"Ejemplo chunk: {len(splits[0].page_content)} chars\n{splits[0].page_content[:200]}...")

✂️ Divididos en 61 chunks
Ejemplo chunk: 764 chars
Trump dice ahora que no descarta desplegar soldados en Irán: "Me dan igual las encuestas" | Internacional



















Es noticiaGuerra IránÚltimas noticiasEEUU RotaTrump IránKuwaitMacron armas ...


## 4. EMBEDDINGS + VECTOR STORE

In [65]:
from langchain_huggingface import HuggingFaceEmbeddings

# all-MiniLM-L6-v2: modelo de embeddings local (~80 MB), rápido y gratuito.
# Convierte cada chunk en un vector de 384 dimensiones que captura su
# significado semántico. Textos similares → vectores cercanos en el espacio.
# Alternativa multilingüe (mejor para español): 'paraphrase-multilingual-MiniLM-L12-v2'
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embeddings listos", embeddings)

✅ Embeddings listos model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [66]:
from langchain_community.vectorstores import FAISS
# FAISS indexa todos los vectores para hacer búsqueda por similitud coseno, y no es persistente (en memoria), 
# pero es muy rápido para prototipos y pruebas locales, con búsquedas en ~1ms, independientemente del número de chunks.
# Para producción multi-usuario considera Chroma (local) o Pinecone (nube).
vectorstore_faiss = FAISS.from_documents(documents=splits, embedding=embeddings)
print("✅ Vectorstore listo (384-dim vectores indexados)")
print(f"   Chunks indexados: {vectorstore_faiss.index.ntotal}")

✅ Vectorstore listo (384-dim vectores indexados)
   Chunks indexados: 61


In [67]:
from langchain_chroma import Chroma
import hashlib

# Al ser persistente, Chroma guarda los vectores en disco en el directorio especificado.
# Si se usa `from_documents()` cada vez, se reindexan todos los documentos duplicándolos,
# porque crea un nuevo índice cada vez que se ejecuta.
# Para evitar duplicados, usamos IDs explícitos basados en un hash del contenido.
# Chroma ignora automáticamente los documentos cuyo ID ya existe en el índice,
# por lo que solo se añaden documentos nuevos sin necesidad de comparar manualmente.

def get_document_hash(document):
    # Hash MD5 único por documento — actúa como ID estable basado en contenido
    return hashlib.md5(document.page_content.encode('utf-8')).hexdigest()

# Carga el vectorstore persistente (o crea uno nuevo si no existe)
vectorstore_chroma = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Genera un ID único por documento basado en su contenido
ids = [get_document_hash(doc) for doc in splits]

# Chroma ignora automáticamente los IDs que ya existen → sin duplicados
vectorstore_chroma.add_documents(documents=splits, ids=ids)
print(f"   Chunks total después de agregar: {vectorstore_chroma._collection.count()}")

   Chunks total después de agregar: 69


## 5. LLM + CADENA RAG

In [68]:
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

def llm_rag_chain(vectorstore):
    # llama-3.3-70b-versatile: mejor modelo del tier gratuito de Groq.
    # temperature=0.1 → respuestas deterministas y fieles al contexto.
    #                   Sube a 0.7+ solo si quieres respuestas más creativas.
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.1
    )

    # El prompt instruye explícitamente al LLM a responder SOLO con el contexto
    # recuperado. Esto es clave para evitar alucinaciones: si la información
    # no está en los chunks, el modelo debe decirlo en vez de inventar.
    prompt_template = """Usa SOLO información de los documentos proporcionados.
    Si no encuentras respuesta, di 'No tengo información suficiente'.

    CONTEXTO:
    {context}

    PREGUNTA: {question}
    RESPUESTA CONCISA:"""
    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    # RetrievalQA encadena los 3 pasos automáticamente:
    #   1. Recupera los k chunks más relevantes del vectorstore
    #   2. Los inyecta en el prompt como {context}
    #   3. Llama al LLM y devuelve la respuesta + fuentes
    #
    # chain_type='stuff': mete todos los chunks en un único prompt.
    # Funciona bien con k<=5. Para documentos muy largos usa 'map_reduce'.
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 4}
        ),
        chain_type_kwargs={"prompt": PROMPT},
        return_source_documents=True  # Devuelve los chunks usados para trazabilidad
    )
    print("🔥 Cadena RAG lista!")
    return qa_chain

In [69]:
qa_chain_faiss = llm_rag_chain(vectorstore_faiss)

🔥 Cadena RAG lista!


In [70]:
qa_chain_chroma = llm_rag_chain(vectorstore_chroma)

🔥 Cadena RAG lista!


## 6. ¡PRUEBAS RAG! 🔥

In [71]:
def rag_test(qa_chain):
    # La búsqueda semántica entiende el significado, no keywords exactos.
    # Puedes preguntar en español aunque los documentos estén en inglés;
    # all-MiniLM-L6-v2 tiene capacidad cross-lingual básica.
    consulta = "¿Qué opina Trump de irán?"
    resultado = qa_chain.invoke({"query": consulta})

    print("🤖 RESPUESTA:")
    print(resultado["result"])
    print("\n" + "="*60)
    print("📄 FUENTES (top-2):")
    for i, doc in enumerate(resultado["source_documents"][:2]):
        print(f"{i+1}. {doc.metadata.get('source', 'Desconocido')[:60]}")
        print(f"   Preview: {doc.page_content[:200]}...\n") 

In [72]:
rag_test(qa_chain_faiss)

🤖 RESPUESTA:
Trump considera que Irán está persiguiendo "siniestras ambiciones" y ha advertido que no descarta el envío de tropas a Irán "si son necesarias", ya que según él, "los estamos destrozando".

📄 FUENTES (top-2):
1. https://www.mundodeportivo.com/actualidad/20260302/100413882
   Preview: . Sobrevivieron el presidente de Irán, Masud Pezeshkian, y Alí Larijani, jefe de Seguridad y nuevo hombre fuerte del régimen de los ayatolás....

2. https://theobjective.com/internacional/2026-03-02/trump-envi
   Preview: Internacional

 
                            Trump no descarta el envío de tropas a Irán «si son necesarias»: «Los estamos destrozando»                        
El presidente de EEUU anuncia que la «gr...



In [73]:
rag_test(qa_chain_chroma)

🤖 RESPUESTA:
Trump no descarta el envío de tropas a Irán "si son necesarias" y afirma que "los estamos destrozando". También anuncia que la "gran ola" de ataques contra Irán "está a punto de llegar".

📄 FUENTES (top-2):
1. https://www.mundodeportivo.com/actualidad/20260302/100413882
   Preview: . Sobrevivieron el presidente de Irán, Masud Pezeshkian, y Alí Larijani, jefe de Seguridad y nuevo hombre fuerte del régimen de los ayatolás....

2. https://theobjective.com/internacional/2026-03-02/trump-envi
   Preview: Internacional

 
                            Trump no descarta el envío de tropas a Irán «si son necesarias»: «Los estamos destrozando»                        
El presidente de EEUU anuncia que la «gr...



## 7. GUARDAR/RECARGAR VECTORSTORE (PRODUCCIÓN)

In [74]:
# Persiste el índice en disco: genera index.faiss (vectores) + index.pkl (metadatos).
# Así evitas re-embeber los documentos cada vez que reinicias el notebook.
vectorstore_faiss.save_local("mi_rag_index")

# allow_dangerous_deserialization=True es requerido porque FAISS usa pickle.
# Solo carga índices que hayas generado tú mismo; nunca de fuentes externas.
vectorstore_reloaded = FAISS.load_local(
    "mi_rag_index", embeddings, allow_dangerous_deserialization=True
)
print("💾 Index guardado y recargado correctamente")
print(f"   Chunks en el índice: {vectorstore_reloaded.index.ntotal}")

💾 Index guardado y recargado correctamente
   Chunks en el índice: 61


In [75]:
# Chroma: Ya está persistente automáticamente. 
# No es necesario usar save/load explícitamente, ni descomentar el bloque de código, 
# ya que de lo contrario estarías duplicando los chunks (ya que se cargan al reiniciar el entorno).
# 
# Si se descomenta, el número de chunks aumentaría innecesariamente.
# 
# vectorstore_chroma_reloaded = Chroma(
#     persist_directory="./chroma_db",
#     embedding_function=embeddings
# )
# print("💾 Chroma recargado (ya persistente)")
# print(f"   Chunks: {vectorstore_chroma_reloaded._collection.count()}")

## 🎯 PRÓXIMOS PASOS

| Mejora              | Cómo                        | Beneficio                                        | Estado |
|---------------------|-----------------------------|--------------------------------------------------|--------|
| **Streaming**        | `qa_chain.stream()`          | Respuestas token a token en tiempo real         | -      |
| **Reranking**        | `CohereRerank`               | Reordena los chunks recuperados → precisión +30% | -      |
| **Cloud DB**         | `Chroma` / `Pinecone`        | Persistencia escalable y multi-usuario           | ✅     |
| **PDFs**             | `PyMuPDFLoader`              | Ingesta documentos empresariales                 | ✅     |
| **Multi-idioma**     | `paraphrase-multilingual-MiniLM-L12-v2` | Embeddings nativos en español y otros idiomas | -      |
| **Evaluación con LangSmith** | `langsmith` + `evaluate()` | Mide la calidad de la cadena RAG con datasets de preguntas/respuestas esperadas... | - |
